# 02 — Simple RAG: Retrieval-Augmented Generation

**RAG** lets you give an LLM access to your own documents so it can answer questions about them — even if the model has never seen that information during training.

How it works:
1. **Index**: Split your documents into chunks, convert each chunk to a vector (embedding)
2. **Retrieve**: When asked a question, find the most relevant chunks via vector similarity
3. **Generate**: Give the retrieved chunks + question to the LLM as context

## Prerequisites
- Ollama running with a model: `ollama pull llama3.2:3b`
- An embedding model: `ollama pull nomic-embed-text` (~274MB, very fast)

## Why RAG? The problem it solves

If you ask `llama3.2:3b` "What is in my company handbook?", it has no idea — that document didn't exist when the model was trained. You could fine-tune the model on your handbook, but that's expensive and slow.

RAG is the cheaper answer: **don't bake the knowledge into the model, just look it up at query time and hand it to the model as context.**

Think of it like an open-book exam. The LLM is the student. Your documents are the textbook. RAG finds the right pages and puts them in front of the student before they answer.

The three steps in plain English:
1. **Before any questions** — convert every document into a vector (a list of numbers) that captures its meaning. Store those vectors.
2. **When a question arrives** — convert the question into a vector the same way, then find which document vectors are closest to it.
3. **Generate** — paste those matching documents into the prompt and ask the LLM to answer using them.

In [1]:
import requests
import json
import numpy as np

OLLAMA_URL = 'http://localhost:11434'
CHAT_MODEL = 'llama3.2:3b'
EMBED_MODEL = 'nomic-embed-text'  # Pull with: ollama pull nomic-embed-text

# Verify Ollama is running
try:
    r = requests.get(f'{OLLAMA_URL}/')
    print('Ollama ready!')
except:
    print('ERROR: Start Ollama first (ollama serve)')

Ollama ready!


## Step 1: Our "Document Database"

In a real RAG system, you'd load PDFs, websites, or files. For this intro, we'll use short text passages about machine learning.

In [2]:
# Our knowledge base — replace these with your own documents!
documents = [
    "PyTorch is an open-source machine learning framework developed by Meta. It uses dynamic computation graphs and is popular for research due to its flexibility and Pythonic design.",
    "Gradient descent is an optimization algorithm that minimizes a loss function by iteratively moving in the direction of the steepest descent as defined by the negative of the gradient.",
    "A convolutional neural network (CNN) is a type of neural network designed for processing grid-like data such as images. It uses convolutional layers to automatically learn spatial features.",
    "Overfitting occurs when a model performs well on training data but poorly on new, unseen data. Common solutions include regularization, dropout, and getting more training data.",
    "Transfer learning is a technique where a model trained on one task is adapted for a different but related task. It allows you to benefit from large pre-trained models even with small datasets.",
    "The attention mechanism, introduced in the paper 'Attention is All You Need', allows models to focus on relevant parts of the input when generating each part of the output. It is the foundation of transformer models.",
    "Ollama is a tool for running large language models locally on your own machine. It supports models like Llama, Gemma, Phi, and Mistral, and provides a simple HTTP API.",
    "RAG (Retrieval-Augmented Generation) combines information retrieval with text generation. It improves LLM responses by grounding them in retrieved documents, reducing hallucinations.",
    "scikit-learn provides simple and efficient tools for machine learning in Python. It includes algorithms for classification, regression, clustering, and dimensionality reduction.",
    "The transformer architecture uses self-attention to process sequences in parallel, unlike RNNs which process sequentially. This makes transformers much faster to train on modern hardware.",
]

print(f'Knowledge base: {len(documents)} documents')

Knowledge base: 10 documents


Each string in `documents` is one "page" in our knowledge base. The model has never seen any of this text during training — we're going to inject it at question time.

Next step: convert each document into a vector so we can do math on it to find relevance.

## Step 2: Create Embeddings

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text.

The key insight: **similar meaning → similar numbers**. If you embed "dog" and "puppy", the resulting vectors are close together. "Dog" and "quantum physics" would be far apart.

We're using a separate model (`nomic-embed-text`) just for this — it's a small model trained specifically to turn text into meaningful vectors. It doesn't generate text, it just produces numbers.

After this step, each of our 10 documents becomes a list of 768 numbers. We can't read those numbers and understand them — but we can do math on them to measure similarity.

In [3]:
def get_embedding(text, model=EMBED_MODEL):
    """Get a vector embedding for a piece of text."""
    response = requests.post(
        f'{OLLAMA_URL}/api/embeddings',
        json={'model': model, 'prompt': text}
    )
    return np.array(response.json()['embedding'])

# Embed all documents (this takes ~10-30 seconds)
print('Creating embeddings...')
doc_embeddings = []
for i, doc in enumerate(documents):
    embedding = get_embedding(doc)
    doc_embeddings.append(embedding)
    print(f'  {i+1}/{len(documents)} done', end='\r')

doc_embeddings = np.array(doc_embeddings)
print(f'\nEmbedding matrix shape: {doc_embeddings.shape}')
print(f'Each document → {doc_embeddings.shape[1]}-dimensional vector')

Creating embeddings...
  10/10 done
Embedding matrix shape: (10, 768)
Each document → 768-dimensional vector


`doc_embeddings` is now a 10×768 matrix — 10 rows (one per document), 768 columns (the numbers representing its meaning).

Think of each document as a point in 768-dimensional space. Documents about similar topics end up near each other in that space. That's the whole trick — "relevant" becomes "nearby."

## Step 3: Semantic Search

Now we need to find which documents are most relevant to a question.

**Cosine similarity** measures the angle between two vectors — not their size, just their direction. Score of 1.0 means pointing in exactly the same direction (identical meaning). Score of 0 means unrelated.

The `retrieve` function does three things:
1. Embed the question (turn it into a 768-number vector)
2. Compare that vector to every document vector using cosine similarity
3. Return the top 3 most similar documents

This is the "Retrieval" part of RAG — finding the right pages from the textbook before handing them to the LLM.

In [4]:
def cosine_similarity(a, b):
    """Measure similarity between two vectors (1 = identical, 0 = unrelated)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=3):
    """Find the top_k most relevant documents for a query."""
    query_embedding = get_embedding(query)
    
    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(documents[i], similarities[i]) for i in top_indices]

# Test retrieval
test_query = 'How do convolutional networks work with images?'
results = retrieve(test_query, top_k=3)

print(f'Query: "{test_query}"')
print()
for i, (doc, score) in enumerate(results):
    print(f'Result {i+1} (similarity: {score:.3f}):')
    print(f'  {doc}')
    print()

Query: "How do convolutional networks work with images?"

Result 1 (similarity: 0.790):
  A convolutional neural network (CNN) is a type of neural network designed for processing grid-like data such as images. It uses convolutional layers to automatically learn spatial features.

Result 2 (similarity: 0.547):
  Transfer learning is a technique where a model trained on one task is adapted for a different but related task. It allows you to benefit from large pre-trained models even with small datasets.

Result 3 (similarity: 0.545):
  PyTorch is an open-source machine learning framework developed by Meta. It uses dynamic computation graphs and is popular for research due to its flexibility and Pythonic design.



Notice the similarity scores:
- **Result 1 (0.790)** — high score, clearly relevant (it's literally about CNNs and images)
- **Results 2 & 3 (~0.547)** — lower scores, only loosely related

The model didn't match on keywords — it matched on *meaning*. If you asked "how do neural nets process pictures?", it would still return the CNN document even though "pictures" doesn't appear in it.

## Step 4: The Full RAG Pipeline

This is where the three pieces connect. `rag_answer` does everything in sequence:

1. **Retrieve** — call `retrieve()` to get the 3 most relevant documents
2. **Build a prompt** — paste those documents into the prompt as "context", then add the question
3. **Generate** — send that prompt to `llama3.2:3b` and return its answer

The critical part is step 2. The LLM never sees your documents directly — it just sees a text prompt that happens to contain the relevant passages. It reads them and answers based on what's there.

Here's what the actual prompt looks like before it's sent to the model:

```
Use the following context to answer the question.
If the answer is not in the context, say so honestly.

Context:
[document 1 text]

[document 2 text]

[document 3 text]

Question: What is transfer learning and when should I use it?

Answer:
```

The model doesn't know these are "retrieved documents" — it just sees text in a prompt and responds to it. RAG is essentially a clever way of automatically writing that context section for you.

In [5]:
def rag_answer(question, top_k=3, model=CHAT_MODEL):
    """Answer a question using retrieved context from our document database."""
    # 1. Retrieve relevant documents
    relevant_docs = retrieve(question, top_k=top_k)
    context = '\n\n'.join([doc for doc, _ in relevant_docs])
    
    # 2. Build the prompt with context
    prompt = f"""Use the following context to answer the question. 
If the answer is not in the context, say so honestly.

Context:
{context}

Question: {question}

Answer:"""
    
    # 3. Generate answer
    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False,
        }
    )
    return response.json()['message']['content'], relevant_docs


# Try it out!
question = 'What is transfer learning and when should I use it?'
answer, sources = rag_answer(question)

print(f'Q: {question}')
print(f'\nA: {answer}')
print('\n--- Sources used ---')
for doc, score in sources:
    print(f'[{score:.2f}] {doc[:80]}...')

Q: What is transfer learning and when should I use it?

A: Transfer learning is a technique where a model trained on one task is adapted for a different but related task. This allows you to benefit from large pre-trained models even with small datasets, potentially improving performance on the target task.

You should use transfer learning when:

- You have access to a large pre-trained model that has already been trained on a similar task.
- You have a small dataset for your specific task and need to leverage the knowledge learned by the pre-trained model.
- You want to adapt a pre-trained model to a new but related task without having to retrain the entire model from scratch.

Keep in mind that transfer learning is not suitable for tasks with significantly different requirements or data distributions compared to the original training task.

--- Sources used ---
[0.72] Transfer learning is a technique where a model trained on one task is adapted fo...
[0.62] PyTorch is an open-source 

In [ ]:
# Try more questions
questions = [
    'What tool can I use to run LLMs locally?',
    'What causes overfitting and how do I fix it?',
    'How does the attention mechanism work?',
]

for q in questions:
    print('=' * 60)
    answer, _ = rag_answer(q)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

## Summary

You've built a complete RAG pipeline from scratch:
- `get_embedding()` — converts text to vectors
- `retrieve()` — finds relevant docs via cosine similarity
- `rag_answer()` — retrieves context, then generates an answer

## What to Try Next

1. **Your own documents**: Replace the `documents` list with text from files you care about
2. **Load from files**: Read `.txt` files from disk using `open()` and `f.read()`
3. **Chunking**: Split large documents into overlapping chunks for better retrieval
4. **FAISS or Chroma**: Replace the brute-force similarity search with a proper vector database for large document sets

```python
# Load a text file
with open('../../data/my_notes.txt') as f:
    text = f.read()

# Simple chunking: split every 500 characters
chunks = [text[i:i+500] for i in range(0, len(text), 400)]
documents = chunks  # replace the documents list above
```